<a href="https://colab.research.google.com/github/vitorjensen/Python_para_Analise_de_Dados/blob/main/Projetos/Ana%CC%81lise%20de%20Comportamento%20dos%20Depo%CC%81sitos%20Totais%20no%20Brasil/Comportamento_de_Dep%C3%B3sitos_Totais_no_Brasil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Resolução das Questões do Case - FGC**

### **Questão 1:**


- Eu considero como sendo possíveis **vantagens**, por exemplo, a segurança que pode ser fornecida para um depositante, evitando que atitudes desesperadas por parte dele sejam tomadas para tentar recuperar o crédito. Uma outra vantagem é o estímulo e a confiança passada ao depositante, para que mantenham seus patrimônios em bancos que sejam autorizados e acompanhados pelo próprio Banco Central.

- Eu considero como sendo possíveis **desvantagens**, por exemplo, bancos que dependem muito de depósitos garantidos pagando taxas altas, podendo oferecer riscos ao fundo. Outra possível desvantagem seria, como valores até o limite de R$250 mil estão protegidos, investidores podem tender a escolher uma instituição bancária apenas pela sua rentabilidade sem avaliar a solidez.

### **Questão 2 - (a)**

#### **Criando os csv's com os dados do BCB e IBGE**

In [3]:
# Importando bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

In [ ]:
# Aqui começa a parte de extrair os dados da fonte BCB

pasta = '/content/drive/MyDrive/'

# Definindo o período correto dos dados

data_inicio = '01/01/2022'
data_fim = '31/12/2025'

# Criando um dicionário para poder correlacionar nome e código das séries
series_sgs = {
    'Poupanca': 1835,
    'Deposito_a_Vista': 27790,
    'Deposito_a_Prazo': 27805,
    'PIB_Mensal': 4380,
    'Selic': 4189,
    'IPCA_12m': 13522,  # Dado para Inflação em 12 meses
    'IPCA_Mensal': 433, # Dado para inflação mês a mês
}

# Criando uma lista vazia

dados_brutos = []

# Laço de repetição que percorre o dicionário primário para que seja feito o request para cada par de "nome" e "código"

for nome, codigo in series_sgs.items():
    url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json&dataInicial={data_inicio}&dataFinal={data_fim}'
    df_series = pd.read_json(url) # Leitura com pandas do conteúdo json

    # Verificação condicional caso alguma série retorne vazia
    if df_series.empty:
      raise ValueError(f'A extração da série {nome} não retornou dados.')

    df_series = df_series.rename(columns={'valor': nome})

    # Alimentando a lista dados_brutos
    dados_brutos.append(df_series)
    print(f'{nome} (SGS {codigo}): {len(df_series)} linhas baixadas')


# Juntando as séries para criar um único CSV, utilizando a coluna 'data' como chave

df_bcb = dados_brutos[0] # Atribuindo o primeiro índice do dicionário de dados ao df_bcb

for df_series in dados_brutos[1:]: # Tecnica de Slice para percorrer a partir do índice 1 e não 0
   df_bcb = df_bcb.merge(df_series, on='data', how='left')

# Salvando a base bruta do BCB
df_bcb.to_csv(pasta + 'bcb.csv', index=False)
print('Base salva como bcb.csv')

In [ ]:
# Aqui começa a parte de extrair os dados da fonte IBGE

# Extraindo a Taxa de desocupação:
url_sidra = 'https://apisidra.ibge.gov.br/values/t/6381/n1/all/v/4099/p/202201-202512'
df_ibge = pd.read_json(url_sidra)

if df_ibge.empty:
  raise ValueError('A extração do IBGE não retornou dados.')

# Salvando a base bruta do IBGE, do jeito que veio
df_ibge.to_csv(pasta + 'ibge.csv', index=False)
print(f'IBGE: {len(df_ibge)} linhas baixadas. Base salva como ibge.csv')

IBGE: 49 linhas baixadas. Base salva como ibge.csv


#### **Etapa de tratamento e limpeza das bases (BCB e IBGE)**

Tratamento e limpeza, base BCB

In [4]:
# Executando Display nas bases

df_bcb = pd.read_csv('/content/drive/MyDrive/bcb.csv')
df_ibge = pd.read_csv('/content/drive/MyDrive/ibge.csv')

#display(df_bcb.head(5))
#display(df_bcb.tail(5))
#display(df_ibge)

In [ ]:
# Visualizando informações das tabelas

df_bcb.info()
df_ibge.info()

In [ ]:
# Renomeando coluna de Data no BCB

df_bcb = df_bcb.rename(columns={'data': 'Data'})

# Convertendo atributo de Texto em Data
df_bcb['Data'] = pd.to_datetime(df_bcb['Data'], format='%d/%m/%Y', errors='coerce')

# Fazendo conferência após conversão
print(df_bcb['Data'].dtype)

# Verificando as datas, mínima e máxima
print(f"Data inicial: {df_bcb['Data'].min()}")
print(f"Data final: {df_bcb['Data'].max()}")

# Fazendo verificação de existência de valores nulos e somando caso exista a ocorrência
print(f"Datas não convertidas: {df_bcb['Data'].isnull().sum()}")

# Verificando a existência de qualquer valor indefinido (NaN, NaT ...)
print(df_bcb['Data'].isna().sum())

Resultado:
- Verificação de total de nulos na base (0)
- Total de linhas e colunas: 48 linhas x 8 colunas
- Conversão do tipo Texto para Data no campo Data
- Modificação nome da coluna Data

Tratamento e Limpeza na base IBGE

In [18]:
# Fazendo a exclusão de colunas indesejadas
colunas_indesejadas = ['NC', 'NN', 'MC', 'MN', 'D1C', 'D1N', 'D2C', 'D2N', 'D3N'] # Lista de colunas
df_ibge = df_ibge.drop(colunas_indesejadas, axis=1, errors='ignore') # axis=1 -> Eixo das colunas
display(df_ibge.head())

,V,D3C
0,Valor,Trimestre Móvel (Código)
1,11.2,202201
2,11.2,202202
3,11.1,202203
4,10.5,202204


Resultado
- Verificação de total de nulos na base (0)
- Total de colunas: 48 linhas x 11 colunas
- Exclusão de colunas indesejadas
- Renomear colunas
- Remoção de linhas de cabeçalho
- Conversão para tipos de Data e Valor nos atributos